# Agente conversacional con memoria y herramientas seguras

### Objetivo

Crear un asistente conversacional con LangChain que:

* use ChatOpenAI(model_name="gpt-4", temperature=0),

* tenga memoria de conversación,

* exponga una tool de cálculo segura (no eval) y otra de utilidades,

## Objetivos


### Objetivo 1

* Inicializa proyecto assistant/ con virtualenv y requirements.txt.

* Crea .env y carga con dotenv.

* Instancia ChatOpenAI(gpt-4, temperature=0).

* Añade ConversationBufferMemory(memory_key="chat_history", return_messages=True).

* Configura un agente initialize_agent(..., agent="conversational-react-description", memory=...).

In [ ]:
# Objetivo 1: Agente conversacional con memoria
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Instanciar el modelo
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

# Configurar memoria de conversación
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Inicializar agente (sin tools por ahora)
agent = initialize_agent(
    tools=[],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

print("Agente conversacional con memoria configurado")
print("\nPrueba de memoria:")
print("-" * 60)

# Primera interacción
response1 = agent.invoke({"input": "Hola, mi nombre es Ana y estudio Inteligencia Artificial"})
print(f"Respuesta 1: {response1['output']}\n")

# Segunda interacción (debe recordar el nombre)
response2 = agent.invoke({"input": "¿Recuerdas cómo me llamo?"})
print(f"Respuesta 2: {response2['output']}")

### Objetivo 2

* Implementa una función que permita realizar operaciones matemáticas básicas de manera segura y controlada.

* Registra la tool como Tool(name="Calculadora", func=<tu_función>, description="Realiza cálculos básicos de forma segura").

In [ ]:
# Objetivo 2: Tool Calculadora segura (sin eval)
import os
import re
import operator
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.memory import ConversationBufferMemory

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

def calculadora_segura(expresion: str) -> str:
    """
    Calcula operaciones matemáticas básicas de forma segura sin usar eval().
    Soporta: suma (+), resta (-), multiplicación (*), división (/).
    
    Args:
        expresion: Expresión matemática en formato "número operador número"
    
    Returns:
        str: Resultado del cálculo o mensaje de error
    """
    try:
        # Limpiar espacios
        expresion = expresion.strip()
        
        # Diccionario de operadores permitidos
        operadores = {
            '+': operator.add,
            '-': operator.sub,
            '*': operator.mul,
            '/': operator.truediv,
            '**': operator.pow
        }
        
        # Patrón para detectar operaciones simples: número operador número
        patron = r'^(-?\d+\.?\d*)\s*([\+\-\*/\*\*]+)\s*(-?\d+\.?\d*)$'
        match = re.match(patron, expresion)
        
        if not match:
            return "Error: Formato no válido. Use: 'número operador número' (ej: '5 + 3')"
        
        num1 = float(match.group(1))
        operador = match.group(2)
        num2 = float(match.group(3))
        
        if operador not in operadores:
            return f"Error: Operador '{operador}' no soportado. Use: +, -, *, /, **"
        
        # Validar división por cero
        if operador == '/' and num2 == 0:
            return "Error: División por cero no permitida"
        
        resultado = operadores[operador](num1, num2)
        
        return f"{expresion} = {resultado}"
    
    except Exception as e:
        return f"Error al calcular: {str(e)}"

# Crear la tool
tool_calculadora = Tool(
    name="Calculadora",
    func=calculadora_segura,
    description="Realiza cálculos matemáticos básicos de forma segura. Formato: 'número operador número'. Operadores: +, -, *, /, **. Ejemplo: '5 + 3'"
)

# Instanciar modelo y memoria
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Inicializar agente con la tool calculadora
agent_con_calc = initialize_agent(
    tools=[tool_calculadora],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

print("Agente con calculadora segura configurado")
print("\nPruebas:")
print("-" * 60)

# Prueba 1
response1 = agent_con_calc.invoke({"input": "¿Cuánto es 25 * 4?"})
print(f"\nResultado 1: {response1['output']}\n")

# Prueba 2
response2 = agent_con_calc.invoke({"input": "Ahora calcula 100 / 5"})
print(f"\nResultado 2: {response2['output']}")

### Objetivo 3

* Implementa una tool convertir_unidades(expr: str) -> str (p. ej., "12 km a m" → "12000 m").

* Define un conjunto mínimo de unidades y valida inputs.

* Documenta en el README los formatos soportados.

In [ ]:
# Objetivo 3: Tool convertir_unidades
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.memory import ConversationBufferMemory

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

def convertir_unidades(expr: str) -> str:
    """
    Convierte unidades de medida entre diferentes sistemas.
    
    Formatos soportados:
    - "valor unidad_origen a unidad_destino" (ej: "10 km a m")
    - "valor unidad_origen to unidad_destino" (ej: "5 kg to g")
    
    Unidades soportadas:
    - Longitud: km, m, cm, mm, mi (millas), ft (pies), in (pulgadas)
    - Masa: kg, g, mg, lb (libras), oz (onzas)
    - Temperatura: C (Celsius), F (Fahrenheit), K (Kelvin)
    
    Args:
        expr: Expresión de conversión
    
    Returns:
        str: Resultado de la conversión o mensaje de error
    """
    try:
        # Patrones para detectar la expresión
        patron1 = r'^(\d+\.?\d*)\s*(\w+)\s+(?:a|to)\s+(\w+)$'
        match = re.match(patron1, expr.strip(), re.IGNORECASE)
        
        if not match:
            return "Error: Formato no válido. Use: 'valor unidad_origen a unidad_destino' (ej: '10 km a m')"
        
        valor = float(match.group(1))
        unidad_origen = match.group(2).lower()
        unidad_destino = match.group(3).lower()
        
        # Diccionario de conversiones a unidad base
        conversiones = {
            # Longitud (base: metros)
            'km': 1000,
            'm': 1,
            'cm': 0.01,
            'mm': 0.001,
            'mi': 1609.34,
            'ft': 0.3048,
            'in': 0.0254,
            
            # Masa (base: gramos)
            'kg': 1000,
            'g': 1,
            'mg': 0.001,
            'lb': 453.592,
            'oz': 28.3495,
        }
        
        # Validar unidades
        if unidad_origen not in conversiones and unidad_origen not in ['c', 'f', 'k']:
            return f"Error: Unidad '{unidad_origen}' no soportada"
        
        if unidad_destino not in conversiones and unidad_destino not in ['c', 'f', 'k']:
            return f"Error: Unidad '{unidad_destino}' no soportada"
        
        # Conversión de temperatura (casos especiales)
        if unidad_origen in ['c', 'f', 'k'] or unidad_destino in ['c', 'f', 'k']:
            return convertir_temperatura(valor, unidad_origen, unidad_destino)
        
        # Validar que las unidades sean del mismo tipo
        longitud_units = {'km', 'm', 'cm', 'mm', 'mi', 'ft', 'in'}
        masa_units = {'kg', 'g', 'mg', 'lb', 'oz'}
        
        if unidad_origen in longitud_units and unidad_destino not in longitud_units:
            return "Error: No se pueden convertir unidades de diferentes categorías"
        if unidad_origen in masa_units and unidad_destino not in masa_units:
            return "Error: No se pueden convertir unidades de diferentes categorías"
        
        # Convertir a unidad base y luego a unidad destino
        valor_base = valor * conversiones[unidad_origen]
        resultado = valor_base / conversiones[unidad_destino]
        
        return f"{valor} {unidad_origen} = {resultado:.4f} {unidad_destino}"
    
    except Exception as e:
        return f"Error al convertir: {str(e)}"

def convertir_temperatura(valor: float, origen: str, destino: str) -> str:
    """Convierte temperaturas entre Celsius, Fahrenheit y Kelvin"""
    origen = origen.lower()
    destino = destino.lower()
    
    # Convertir primero a Celsius
    if origen == 'c':
        celsius = valor
    elif origen == 'f':
        celsius = (valor - 32) * 5/9
    elif origen == 'k':
        celsius = valor - 273.15
    else:
        return f"Error: Unidad de temperatura '{origen}' no soportada"
    
    # Convertir de Celsius a destino
    if destino == 'c':
        resultado = celsius
    elif destino == 'f':
        resultado = (celsius * 9/5) + 32
    elif destino == 'k':
        resultado = celsius + 273.15
    else:
        return f"Error: Unidad de temperatura '{destino}' no soportada"
    
    return f"{valor} {origen.upper()} = {resultado:.2f} {destino.upper()}"

# Crear las tools
tool_calculadora = Tool(
    name="Calculadora",
    func=calculadora_segura,
    description="Realiza cálculos matemáticos básicos de forma segura. Formato: 'número operador número'. Ejemplo: '5 + 3'"
)

tool_conversor = Tool(
    name="ConvertirUnidades",
    func=convertir_unidades,
    description="Convierte unidades de medida. Formato: 'valor unidad_origen a unidad_destino'. Soporta: km,m,cm,mm,mi,ft,in (longitud); kg,g,mg,lb,oz (masa); C,F,K (temperatura). Ejemplo: '10 km a m'"
)

# Instanciar modelo y memoria
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Inicializar agente con ambas tools
agent_completo = initialize_agent(
    tools=[tool_calculadora, tool_conversor],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

print("Agente con calculadora y conversor de unidades configurado")
print("\nPruebas:")
print("-" * 60)

# Prueba 1: Conversión
response1 = agent_completo.invoke({"input": "Convierte 5 km a metros"})
print(f"\nResultado 1: {response1['output']}\n")

# Prueba 2: Temperatura
response2 = agent_completo.invoke({"input": "¿Cuántos grados Fahrenheit son 25 grados Celsius?"})
print(f"\nResultado 2: {response2['output']}")

### Objetivo 4

* Activa streaming=True con StreamingStdOutCallbackHandler.

* Demuestra en consola la emisión progresiva de tokens.

In [ ]:
# Objetivo 4: Streaming de tokens con StreamingStdOutCallbackHandler
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.memory import ConversationBufferMemory
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Cargar configuración
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Reutilizar las funciones de tools anteriores
# (calculadora_segura y convertir_unidades ya están definidas)

# Crear las tools
tool_calculadora = Tool(
    name="Calculadora",
    func=calculadora_segura,
    description="Realiza cálculos matemáticos básicos de forma segura. Formato: 'número operador número'. Ejemplo: '5 + 3'"
)

tool_conversor = Tool(
    name="ConvertirUnidades",
    func=convertir_unidades,
    description="Convierte unidades de medida. Formato: 'valor unidad_origen a unidad_destino'. Soporta: km,m,cm,mm,mi,ft,in (longitud); kg,g,mg,lb,oz (masa); C,F,K (temperatura). Ejemplo: '10 km a m'"
)

# Instanciar modelo con streaming activado
llm_streaming = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# Configurar memoria
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Inicializar agente con streaming
agent_streaming = initialize_agent(
    tools=[tool_calculadora, tool_conversor],
    llm=llm_streaming,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)

print("="*60)
print("AGENTE CON STREAMING ACTIVADO")
print("="*60)
print("\nObserva cómo los tokens se emiten progresivamente...\n")
print("-"*60)

# Demostración de streaming
print("\nPregunta 1: Explicación general")
print("-"*60)
response1 = agent_streaming.invoke({
    "input": "Explícame brevemente qué es la inteligencia artificial y dame un ejemplo de aplicación"
})
print(f"\n\n[Respuesta completa guardada en memoria]\n")

print("\n" + "="*60)
print("\nPregunta 2: Usando tools con streaming")
print("-"*60)
response2 = agent_streaming.invoke({
    "input": "Calcula cuántos metros hay en 2.5 kilómetros y luego multiplica ese resultado por 3"
})

print("\n\n" + "="*60)
print("RESUMEN")
print("="*60)
print("\nEl streaming permite ver los tokens generados en tiempo real,")
print("mejorando la experiencia del usuario en conversaciones largas.")
print("\nEl agente mantiene:")
print("  ✓ Memoria de conversación")
print("  ✓ Acceso a tools (Calculadora y Conversor)")
print("  ✓ Emisión progresiva de tokens (streaming)")
print("="*60)